In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from scipy import stats
import warnings

df = pd.read_parquet("/Users/rohanaryagondi/Library/CloudStorage/OneDrive-YaleUniversity/Yale/Courses/Spring 2026/Beng 2800/Project/Claude/data/final/orientation_neuron_analysis.parquet")
t = np.load("/Users/rohanaryagondi/Library/CloudStorage/OneDrive-YaleUniversity/Yale/Courses/Spring 2026/Beng 2800/Project/Claude/data/processed/orientation_binned_tuning.npz")
tmean, tsem, cdeg = t["tuning_mean"], t["tuning_sem"], t["angle_bin_centers_deg"]
crad = np.radians(cdeg)

# von Mises tuning model
def vm(theta, b, a, k, t0):
    return b + a * np.exp(k * np.cos(2 * (theta - t0)))

In [ ]:
dff = df[df["fit_success"]]
tf = np.linspace(0, np.pi, 200)

# sample 2 neurons from each R^2 quality bin
fig, axes = plt.subplots(2, 4, figsize=(14, 6))
for i, (lo, hi) in enumerate([(0.9, 1.01), (0.7, 0.9), (0.5, 0.7), (0.3, 0.5)]):
    sub = dff[(dff["fit_r2"] >= lo) & (dff["fit_r2"] < hi)]
    picks = sub.sample(2, random_state=42)
    for j, (idx, row) in enumerate(picks.iterrows()):
        ax = axes[j, i]
        ax.errorbar(cdeg, tmean[idx], yerr=tsem[idx], fmt="o", ms=3, capsize=2, color="steelblue")
        ax.plot(np.degrees(tf), vm(tf, row.fit_baseline, row.fit_amplitude,
                row.fit_kappa_or_width, np.radians(row.fit_pref_orientation_deg)),
                color="tomato", lw=2)
        ax.set_title(f"\u03ba={row.fit_kappa_or_width:.1f}, R\u00b2={row.fit_r2:.2f}", fontsize=9)
plt.tight_layout()

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
ax1.hist(dff["fit_pref_orientation_deg"], bins=36, color="mediumpurple", edgecolor="white")
ax1.axhline(len(dff) / 36, color="red", ls="--")  # uniform reference
ax1.set_xlabel("Preferred orientation (\u00b0)")
ax2.hist(dff["fit_kappa_or_width"], bins=60, color="coral", edgecolor="white")
ax2.set_xlabel("\u03ba")
plt.tight_layout()

In [ ]:
dv = df[df["fit_success"] & df["split_half_reliability"].notna()]
k = dv["fit_kappa_or_width"].values
r = dv["split_half_reliability"].values
m = dv["mean_response"].values
n = len(dv)

# OLS: reliability ~ intercept + kappa + mean_response
X = np.column_stack([np.ones(n), k, m])
beta, _, _, _ = np.linalg.lstsq(X, r, rcond=None)
resid = r - X @ beta

# standard errors, t-stats, p-values
sig2 = np.sum(resid**2) / (n - 3)
se = np.sqrt(sig2 * np.diag(np.linalg.inv(X.T @ X)))
tstat = beta / se
pval = 2 * (1 - stats.t.cdf(np.abs(tstat), df=n - 3))

for name, b, s, t, p in zip(["intercept", "kappa", "mean_response"], beta, se, tstat, pval):
    print(f"{name:>15}: \u03b2={b:.6f}  SE={s:.6f}  t={t:.1f}  p={p:.1e}")

rs, _ = stats.spearmanr(k, r)
print(f"\nSpearman r = {rs:.3f}")

In [ ]:
rng = np.random.default_rng(42)
nsim = 5000
crad_sim = np.radians(np.linspace(2.5, 177.5, 36))

# simulate neurons with noise independent of kappa
true_k = np.clip(rng.exponential(4, nsim), 0, 20)
true_t0 = rng.uniform(0, np.pi, nsim)
true_b = rng.uniform(1, 10, nsim)
true_a = rng.uniform(0.5, 5, nsim)
noise = np.abs(rng.normal(3, 1.5, nsim))

# split-half reliability and refit kappa per simulated neuron
sim_k, sim_r = np.empty(nsim), np.empty(nsim)
for i in range(nsim):
    tc = vm(crad_sim, true_b[i], true_a[i], true_k[i], true_t0[i])
    se_i = noise[i] / np.sqrt(64)
    h1 = tc + rng.normal(0, se_i, 36)
    h2 = tc + rng.normal(0, se_i, 36)
    sim_r[i], _ = stats.pearsonr(h1, h2)
    try:
        mn = (h1 + h2) / 2
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            popt, _ = curve_fit(vm, crad_sim, mn,
                p0=[np.percentile(mn, 10), max(mn.max() - np.percentile(mn, 10), 0.01),
                    1.0, crad_sim[np.argmax(mn)]],
                bounds=([-np.inf, 0, 0, -np.inf], [np.inf, np.inf, 20, np.inf]),
                maxfev=5000)
        sim_k[i] = popt[2]
    except:
        sim_k[i] = sim_r[i] = np.nan

ok = ~np.isnan(sim_k) & ~np.isnan(sim_r)
sk, sr = sim_k[ok], sim_r[ok]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
axes[0].scatter(sk, sr, s=2, alpha=0.3, color="#e6550d", rasterized=True)
axes[0].set_title(f"Simulation\nSpearman r = {stats.spearmanr(sk, sr)[0]:.3f}")
axes[0].set_xlabel("Fitted \u03ba"); axes[0].set_ylabel("Reliability")
axes[0].set_xlim(0, 20); axes[0].set_ylim(-0.5, 1.05)

axes[1].scatter(k, r, s=1, alpha=0.1, color="#2171b5", rasterized=True)
axes[1].set_title(f"Real data\nSpearman r = {rs:.3f}")
axes[1].set_xlabel("Fitted \u03ba"); axes[1].set_ylabel("Reliability")
axes[1].set_xlim(0, 20); axes[1].set_ylim(-0.5, 1.05)

# binned mean reliability vs kappa
edges = np.linspace(0, 20, 11)
bc = (edges[:-1] + edges[1:]) / 2
for dk, dr, lbl, c in [(sk, sr, "Simulation", "#e6550d"), (k, r, "Real", "#2171b5")]:
    bm, bs = [], []
    for j in range(10):
        mask = (dk >= edges[j]) & (dk < edges[j + 1])
        vals = dr[mask]
        bm.append(vals.mean() if len(vals) else np.nan)
        bs.append(vals.std() / np.sqrt(len(vals)) if len(vals) > 1 else np.nan)
    axes[2].errorbar(bc, bm, yerr=bs, fmt="o-", color=c, capsize=3, label=lbl)
axes[2].set_xlabel("\u03ba"); axes[2].set_ylabel("Mean reliability")
axes[2].legend()
plt.tight_layout()